# PIDNet

PIDNet es una arquitectura para segmetación semántica en tiempo real que emplea el concepto básico de un controlado PID (Proportional-Integral-Derivative). Muchos modelos con gran balance de exactitud y velocidad adecuados para segmentación en tiempo real (BiSeNet, Fast-SCNN, DDRNet, etc) son tipo TBN (Two-Branch-Network). Al analizar esta estrategia desde la perspectiva de un controlador PID, se puede hacer una equivalencia de dicha arquitectura con un controlador PI, el cuál suele tener problemas de sobrepaso (overshoot). A nivel imagen, esto puede traducirse en una rama que aprovecha la información semántica original (P) y otra rama que almacena información contextual de baja frecuencia (I), tal que un modelo de segmentación que usa la fusión de ambas conlleva el riesgo de que los límites de los objetos se vean excesivamente erosionados por los pixeles circundantes y que los objetos pequeños quede eclipsados por los objetos grandes adyacentes. 

Para mitigar eso, se propone adjuntar una tercera rama "derivativa". La rama derivativa de un controlador PID se enfoca en la velocidad de cambio de una señal, permitiendo una mejor sensibilidad a cambios de alta frecuencia. En una imagen, esto representa un mejor enfoque sobre los bordes de los objetos o regiones. Dado que las grietas suelen tener una elevada relación perímetro-área, una red cuya detección de bordes está mejorada, en teoría resulta ideal para una aplicación de segmentación de grietas.

La red propuesta, además utiliza los siguientes módulos específicos para que las ramas interactúen de forma inteligente:
- Pag (Pixel-attention-guided fusion): Este módulo permite que la rama de detalles (P) aprenda selectivamente características semánticas ricas de la rama de contexto (I) sin ser abrumada por ella. Utiliza un mecanismo de atención por píxel para decidir cuánta confianza otorgar a la información de contexto en cada punto.
- PAPPM (Parallel Aggregation Pyramid Pooling Module): Es una versión optimizada del módulo PPM tradicional. En lugar de procesar las escalas de forma secuencial, lo hace de forma paralela para reducir la latencia y mantener la velocidad necesaria en aplicaciones de tiempo real.
- Bag (Boundary-attention-guided fusion): Es el corazón de la integración PID. Utiliza las fronteras detectadas por la rama D para guiar la fusión entre la rama de detalles (P) y la de contexto (I). Básicamente, "obliga" a la red a confiar más en la rama de detalles cuando está cerca de un borde y en la de contexto cuando está dentro de un objeto.

La función de pérdida de la red resulta más compleja. Se compone una suma ponderada de 4 pérdidas que balancean el aprendizaje de bordes y semántica. 

$$ Loss = \lambda_0l_0 + \lambda_1l_1 + \lambda_2l_2 + \lambda_3l_3$$
1. $l0$: Pérdida semántica auxiliar para optimizar toda la red
2. $l1$: Pérdida de entropía cruzada binaria ponderada para la detección de bordes (rama D)
3. $l2$: Pérdida de entropía cruzada estándar para la segmentación final
4. $l3$: Pérdida de entropía cruzada con conciencia de frontera (boundary-awareness), que coordina las tareas de segmentación y detección de bordes

**Adaptación para rama derivativa**: La rama derivativa requiere una supervisión directa mediante mapas de bordes reales para entrenarse correctamente. Dado que el dataset de DeepCrack proporciona únicamente las máscaras binarias grieta/fondo, es necesario generar también las etiquetas de borde. Para ello, se propone generarlas usando un detector de bordes de Canny durante la misma carga del dataset (on-the-fly).

In [1]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision import transforms


In [ ]:
class DeepCrackDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        """
        split: 'train' o 'test'
        """
        self.root_dir = root_dir
        self.img_dir = os.path.join(self.root_dir, f"{split}_img")
        self.lab_dir = os.path.join(self.root_dir, f"{split}_lab")
        self.images = sorted(os.listdir(self.img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def _generate_boundary_on_the_fly(self, mask):
        # Habría que jugar un poco con los parámetros de OpenCV para optimizar resultados,
        # pero probablemente lo mejor sería un tratamiento por imagen individual
        # Aplicar Canny para detectar los bordes
        edges = cv2.Canny(mask, 10, 100)

        # Dilatación morfológica para expandir fronteras y conpensar ruido
        kernel = np.ones((3, 3), np.uint8)
        dilated_edges = cv2.dilate(edges, kernel, interations=1)

        # Normalizar 0-1
        return (dilated_edges > 0).astype(np.float32)
